# Forecast clase A: ETS vs media móvil

La media móvil 30 días es el baseline de todo el catálogo. En el **top 100 clase A** (más ventas, series más regulares) se prueba Holt-Winters (ETS): nivel, tendencia amortiguada y estacionalidad semanal.

1. Mismo holdout de 30 días que la notebook 02
2. MAE por SKU y global (MA30 vs ETS)
3. El CLI `predict` usa ETS donde el modelo ajusta; el resto sigue en media móvil


In [ ]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "inventario_ecommerce").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions, save_processed
from inventario_ecommerce.features import (
    clean_transactions,
    compute_abc_classification,
    prepare_daily_demand,
    sales_by_product_last_quarter,
)
from inventario_ecommerce.modeling.ets import temporal_backtest_ets_vs_baseline
from inventario_ecommerce.plots import plot_class_a_mae_comparison

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Top-N clase A:", config.CLASS_A_ETS_TOP_N)
print("Estacionalidad ETS:", config.ETS_SEASONAL_PERIODS, "días")


## 1. Datos

Misma higiene de demanda que el resto del pipeline (`prepare_daily_demand`).


In [ ]:
raw = load_transactions()
clean = clean_transactions(raw)
abc = compute_abc_classification(sales_by_product_last_quarter(clean))
daily = prepare_daily_demand(clean)

print(f"SKUs clase A (trimestre): {(abc['ABCClass'] == 'A').sum():,}")
print(f"Días-SKU calendario: {len(daily):,}")


## 2. Backtest MA30 vs ETS

Holt-Winters aditivo, tendencia amortiguada, periodo 7. Si el ajuste falla o hay menos de 56 días de historia, se usa la media móvil (fallback).


In [ ]:
by_sku, global_metrics = temporal_backtest_ets_vs_baseline(daily, abc)
global_metrics


In [ ]:
print(
    "ETS gana en"
    f" {int(global_metrics.iloc[0]['SKUsETSBetter'])}"
    f" / {int(global_metrics.iloc[0]['SKUsEval'])} SKUs"
)
by_sku.sort_values("MAE_ma30", ascending=False).head(15)


In [ ]:
save_processed(by_sku, "forecast_backtest_class_a.csv")
save_processed(global_metrics, "forecast_backtest_class_a_global.csv")
plot_class_a_mae_comparison(global_metrics, save=True)


El CLI `predict` sustituye la media móvil por ETS en esos SKUs clase A cuando el modelo ajusta. B y C (y el resto de A) siguen en el baseline.
